In [1]:
import pandas as pd
import numpy as np
import glob

In [2]:
files = glob.glob("../data/raw/*.csv")
print("CSV files found:", len(files))


CSV files found: 8


In [3]:
# read  all csv files

dataframes=[]

for file in files:
    temp_df = pd.read_csv(file)
    temp_df.columns = temp_df.columns.str.strip()
    dataframes.append(temp_df)


# Combine all files

df = pd.concat(dataframes,ignore_index=True)

print("Combined dataset shape:", df.shape)
print(df["Label"].value_counts())

Combined dataset shape: (2830743, 79)
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [4]:
# Replace infinite values with missing values
df = df.replace([np.inf, -np.inf], np.nan)

# Remove rows with missing values
df = df.dropna()

print("Dataset shape after cleaning:", df.shape)

Dataset shape after cleaning: (2827876, 79)


In [5]:
# 0 = Benign, 1 = Attack
df["Label"] = df["Label"].astype(str).str.strip()

df["Label"] = df["Label"].apply(
    lambda label: 0 if label.upper() == "BENIGN" else 1
)

print(df["Label"].value_counts())

Label
0    2271320
1     556556
Name: count, dtype: int64


In [6]:
X = df.select_dtypes(include="number").drop(
     columns=["Label"], errors="ignore"
)
y = df["Label"]

print("Input features:", X.shape)
print("Target labels:", y.shape)

Input features: (2827876, 78)
Target labels: (2827876,)


In [7]:
sample_df=df.sample(n=min(100000, len(df)),
    random_state=42
)

X = sample_df.select_dtypes(include="number").drop(
    columns=["Label"], errors="ignore"
)

y = sample_df["Label"]

print("Sample size:", X.shape)
print(y.value_counts())

Sample size: (100000, 78)
Label
0    80252
1    19748
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 80000
Testing rows: 20000


In [9]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [10]:
print("Number of model features:", len(X_train.columns))

print("\nExpected feature names:")
print(X_train.columns.tolist())

Number of model features: 78

Expected feature names:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count

In [14]:
import pandas as pd

live_flows = pd.read_csv(
    r"D:\project\explainable-network-threat-intelligence\data\processed\live_flows.csv"
)

feature_mapping = {
    "Destination Port": "dst_port",
    "Flow Duration": "flow_duration",
    "Total Fwd Packets": "tot_fwd_pkts",
    "Total Backward Packets": "tot_bwd_pkts",
    "Total Length of Fwd Packets": "totlen_fwd_pkts",
    "Total Length of Bwd Packets": "totlen_bwd_pkts",
    "Fwd Packet Length Max": "fwd_pkt_len_max",
    "Fwd Packet Length Min": "fwd_pkt_len_min",
    "Fwd Packet Length Mean": "fwd_pkt_len_mean",
    "Fwd Packet Length Std": "fwd_pkt_len_std",
    "Bwd Packet Length Max": "bwd_pkt_len_max",
    "Bwd Packet Length Min": "bwd_pkt_len_min",
    "Bwd Packet Length Mean": "bwd_pkt_len_mean",
    "Bwd Packet Length Std": "bwd_pkt_len_std",
    "Flow Bytes/s": "flow_byts_s",
    "Flow Packets/s": "flow_pkts_s",
    "Flow IAT Mean": "flow_iat_mean",
    "Flow IAT Std": "flow_iat_std",
    "Flow IAT Max": "flow_iat_max",
    "Flow IAT Min": "flow_iat_min",
    "Fwd IAT Total": "fwd_iat_tot",
    "Fwd IAT Mean": "fwd_iat_mean",
    "Fwd IAT Std": "fwd_iat_std",
    "Fwd IAT Max": "fwd_iat_max",
    "Fwd IAT Min": "fwd_iat_min",
    "Bwd IAT Total": "bwd_iat_tot",
    "Bwd IAT Mean": "bwd_iat_mean",
    "Bwd IAT Std": "bwd_iat_std",
    "Bwd IAT Max": "bwd_iat_max",
    "Bwd IAT Min": "bwd_iat_min",
    "Fwd PSH Flags": "fwd_psh_flags",
    "Bwd PSH Flags": "bwd_psh_flags",
    "Fwd URG Flags": "fwd_urg_flags",
    "Bwd URG Flags": "bwd_urg_flags",
    "Fwd Header Length": "fwd_header_len",
    "Bwd Header Length": "bwd_header_len",
    "Fwd Packets/s": "fwd_pkts_s",
    "Bwd Packets/s": "bwd_pkts_s",
    "Min Packet Length": "pkt_len_min",
    "Max Packet Length": "pkt_len_max",
    "Packet Length Mean": "pkt_len_mean",
    "Packet Length Std": "pkt_len_std",
    "Packet Length Variance": "pkt_len_var",
    "FIN Flag Count": "fin_flag_cnt",
    "SYN Flag Count": "syn_flag_cnt",
    "RST Flag Count": "rst_flag_cnt",
    "PSH Flag Count": "psh_flag_cnt",
    "ACK Flag Count": "ack_flag_cnt",
    "URG Flag Count": "urg_flag_cnt",
    "CWE Flag Count": "cwr_flag_count",
    "ECE Flag Count": "ece_flag_cnt",
    "Down/Up Ratio": "down_up_ratio",
    "Average Packet Size": "pkt_size_avg",
    "Avg Fwd Segment Size": "fwd_seg_size_avg",
    "Avg Bwd Segment Size": "bwd_seg_size_avg",
    "Fwd Header Length.1": "fwd_header_len",
    "Fwd Avg Bytes/Bulk": "fwd_byts_b_avg",
    "Fwd Avg Packets/Bulk": "fwd_pkts_b_avg",
    "Fwd Avg Bulk Rate": "fwd_blk_rate_avg",
    "Bwd Avg Bytes/Bulk": "bwd_byts_b_avg",
    "Bwd Avg Packets/Bulk": "bwd_pkts_b_avg",
    "Bwd Avg Bulk Rate": "bwd_blk_rate_avg",
    "Subflow Fwd Packets": "subflow_fwd_pkts",
    "Subflow Fwd Bytes": "subflow_fwd_byts",
    "Subflow Bwd Packets": "subflow_bwd_pkts",
    "Subflow Bwd Bytes": "subflow_bwd_byts",
    "Init_Win_bytes_forward": "init_fwd_win_byts",
    "Init_Win_bytes_backward": "init_bwd_win_byts",
    "act_data_pkt_fwd": "fwd_act_data_pkts",
    "min_seg_size_forward": "fwd_seg_size_min",
    "Active Mean": "active_mean",
    "Active Std": "active_std",
    "Active Max": "active_max",
    "Active Min": "active_min",
    "Idle Mean": "idle_mean",
    "Idle Std": "idle_std",
    "Idle Max": "idle_max",
    "Idle Min": "idle_min"
}

missing_features = [
    feature for feature in X_train.columns
    if feature not in feature_mapping
]

missing_columns = [
    column for column in feature_mapping.values()
    if column not in live_flows.columns
]

print("Expected model features:", len(X_train.columns))
print("Mapped features:", len(feature_mapping))
print("Missing model features:", missing_features)
print("Missing CSV columns:", missing_columns)

Expected model features: 78
Mapped features: 78
Missing model features: []
Missing CSV columns: []


In [15]:
print(X_train[
    ["Flow Duration", "Flow Bytes/s", "Flow Packets/s"]
].head())

print("\nTraining data statistics:")
print(X_train[
    ["Flow Duration", "Flow Bytes/s", "Flow Packets/s"]
].describe())

         Flow Duration   Flow Bytes/s  Flow Packets/s
1187747          61229    4442.339414       65.328521
1056347          30544    5107.386066      130.958617
2054513             47       0.000000    42553.191490
382958              23  260869.565200    86956.521740
1133982          49103    4765.492943       81.461418

Training data statistics:
       Flow Duration  Flow Bytes/s  Flow Packets/s
count   8.000000e+04  8.000000e+04    8.000000e+04
mean    1.481916e+07  1.454011e+06    7.087080e+04
std     3.364911e+07  2.551617e+07    2.561158e+05
min    -1.000000e+00 -1.200000e+07   -2.000000e+06
25%     1.550000e+02  1.190567e+02    3.187183e+00
50%     3.146600e+04  4.571890e+03    1.042248e+02
75%     3.599602e+06  1.632653e+05    2.312139e+04
max     1.200000e+08  2.070000e+09    3.000000e+06


In [16]:
live_features = pd.DataFrame()

for model_feature, live_column in feature_mapping.items():
    live_features[model_feature] = live_flows[live_column]

print("Live traffic statistics:")

print(
    live_features[
        ["Flow Duration", "Flow Bytes/s", "Flow Packets/s"]
    ].describe()
)

Live traffic statistics:
       Flow Duration  Flow Bytes/s  Flow Packets/s
count      17.000000     17.000000       17.000000
mean        0.083839   4572.723985       36.994516
std         0.087827   8686.629403       14.048013
min         0.000000      0.000000        0.000000
25%         0.070687   2221.856485       40.712744
50%         0.072559   3002.068993       41.249588
75%         0.073466   3072.621832       42.213092
max         0.411716  38009.210232       48.577175


In [11]:
from sklearn.metrics import accuracy_score, classification_report
# Predict test data
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)

# Detailed results
print(classification_report(
    y_test,
    y_pred,
    target_names=["Benign", "Attack"],
    zero_division=0
))

Model Accuracy: 0.99705
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00     16050
      Attack       1.00      0.99      0.99      3950

    accuracy                           1.00     20000
   macro avg       1.00      0.99      1.00     20000
weighted avg       1.00      1.00      1.00     20000



In [17]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import accuracy_score, classification_report

In [24]:
preprocessed_data = joblib.load(
    "../data/processed/preprocessed_data.pkl"
)

X_train = preprocessed_data["x_train"]
X_test = preprocessed_data["x_test"]

y_train = preprocessed_data["y_train"]
y_test = preprocessed_data["y_test"]

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (80000, 78)
Testing data: (20000, 78)


In [12]:
import os
import joblib

# Create models folder
os.makedirs("../models", exist_ok=True)

# Save trained model
joblib.dump(model, "../models/network_model.joblib")

# Save feature column names
joblib.dump(X.columns.tolist(), "../models/feature_columns.joblib")

# Save sample training and testing data for SHAP and LIME
joblib.dump(X_train.head(500), "../models/x_train_sample.joblib")
joblib.dump(X_test.head(100), "../models/x_test_sample.joblib")

# Save complete test data for evaluation and uncertainty analysis
joblib.dump(X_test, "../models/X_test.joblib")
joblib.dump(y_test, "../models/y_test.joblib")

print("Model and all required data saved successfully!")

Model and all required data saved successfully!


In [25]:
import os
import joblib

# Create models folder
os.makedirs("../models", exist_ok=True)

# Save trained model
joblib.dump(model, "../models/network_model.joblib")

# Save feature column names
joblib.dump(
    X_train.columns.tolist(),
    "../models/feature_columns.joblib"
)

# Save sample training data for SHAP and LIME
joblib.dump(X_train.head(500), "../models/x_train_sample.joblib")
joblib.dump(X_test.head(100), "../models/x_test_sample.joblib")

# Save test data for evaluation and uncertainty analysis
joblib.dump(X_test, "../models/X_test.joblib")
joblib.dump(y_test, "../models/y_test.joblib")

print("Model and all required data saved successfully!")

Model and all required data saved successfully!


In [26]:
print(os.listdir("../models"))

['evaluation_results.csv', 'feature_columns.joblib', 'network_model.joblib', 'README.md', 'uncertainty_results.csv', 'X_test.joblib', 'x_test_sample.joblib', 'x_train_sample.joblib', 'y_test.joblib']


In [27]:
import os

print(os.listdir("../models"))

['evaluation_results.csv', 'feature_columns.joblib', 'network_model.joblib', 'README.md', 'uncertainty_results.csv', 'X_test.joblib', 'x_test_sample.joblib', 'x_train_sample.joblib', 'y_test.joblib']


In [13]:
print(os.path.exists("../models/x_train_sample.joblib"))
print(os.path.exists("../models/x_test_sample.joblib"))

True
True


In [21]:
import os

print("Current folder:", os.getcwd())

print("\nFiles inside processed folder:")

print(os.listdir("../data/processed"))

Current folder: d:\project\explainable-network-threat-intelligence\notebooks

Files inside processed folder:
['live_flows.csv', 'live_traffic.pcap', 'README.md']


In [23]:
import os
import joblib

# Create processed folder
os.makedirs("../data/processed", exist_ok=True)

# Save preprocessed data
preprocessed_data = {
    "x_train": X_train,
    "x_test": X_test,
    "y_train": y_train,
    "y_test": y_test
}

joblib.dump(
    preprocessed_data,
    "../data/processed/preprocessed_data.pkl"
)

print("Preprocessed data saved successfully!")

Preprocessed data saved successfully!
